In [199]:
import pandas as pd 
import altair as alt

In [200]:
data = pd.read_excel('../data/raw/final_data.xlsx', sheet_name='Student_Observations')
initial_model_output = pd.read_csv('../data/output/group_initial_model.csv')

`initial_model_output` is the format output of the BKT model, where each row gives the estimates for each KC for each student.

- `correct_predictions` : predicted probability that student answers that specific attempt correctly
- `state_predictions` : predicted probability that the student has mastered the skill at that point in time (P(Learned))

In [201]:
initial_model_output

,Unnamed: 0,user_id,skill_name,correct,order_id,correct_predictions,state_predictions
0,73,S001,MKC.U3.05.blocking_matching_blinding_confounding,1,0,0.439994,0.172071
1,87,S001,MKC.U8.02.chi_square_two_way_structure,0,1,0.397735,0.484439
2,95,S001,MKC.U4.04.binomial_model,0,2,0.443429,0.413403
3,96,S001,MKC.U6.02.one_prop_hypothesis_tests,0,3,0.393887,0.300741
4,121,S001,MKC.U9.01.slope_inference_foundations_ci,0,4,0.480459,0.701718
...,...,...,...,...,...,...,...
1958,1902,S025,MKC.U2.03.correlation_form_outliers_transform,1,90,0.618641,0.872260
1959,1908,S025,MKC.U6.01.one_prop_confidence_intervals,0,91,0.422965,0.611754
1960,1914,S025,MKC.U6.03.errors_alpha_power,1,92,0.417391,0.340129
1961,1929,S025,MKC.U10.02.bootstrap_intervals,0,93,0.279051,0.072029


## Distribution of KC mastery

It could be useful for a teacher to see how the distribution of students on KCs.

In [202]:
initial_model_output.sort_values(['skill_name','user_id'])

,Unnamed: 0,user_id,skill_name,correct,order_id,correct_predictions,state_predictions
69,1598,S001,MKC.U1.01.data_context_variables,0,69,0.572580,0.502561
102,442,S002,MKC.U1.01.data_context_variables,1,11,0.572580,0.502561
120,802,S002,MKC.U1.01.data_context_variables,1,29,0.597781,0.655759
226,1841,S003,MKC.U1.01.data_context_variables,1,68,0.572580,0.502561
252,757,S004,MKC.U1.01.data_context_variables,0,19,0.572580,0.502561
...,...,...,...,...,...,...,...
1401,559,S019,MKC.U9.03.slope_hypothesis_tests,1,21,0.396837,0.259181
1499,799,S020,MKC.U9.03.slope_hypothesis_tests,0,34,0.396837,0.259181
1602,1300,S021,MKC.U9.03.slope_hypothesis_tests,1,46,0.396837,0.259181
1775,1325,S023,MKC.U9.03.slope_hypothesis_tests,0,58,0.396837,0.259181


In [203]:
threshold = 0.95

hist = alt.Chart(initial_model_output).transform_calculate(
    zone=f"""
        datum.state_predictions >= {threshold} ? 'Mastered' :
        datum.state_predictions >= 0.4 ? 'Developing' : 'At risk'
    """
).mark_bar().encode(
    x=alt.X('state_predictions:Q', bin=alt.Bin(maxbins=30), title='Mastery Probability'),
    y=alt.Y('count()', title='Count'),
    color=alt.Color('zone:N', scale=alt.Scale(
        domain=['At risk', 'Developing', 'Mastered'],
        range=['#e57373', '#f0a500', '#4caf77']
    ), legend=alt.Legend(title='Zone'))
)

# Mastery threshold line
rule = alt.Chart(pd.DataFrame({'x': [threshold]})).mark_rule(
    strokeDash=[4, 4], color='gray', size=1.5
).encode(x='x:Q')

(hist + rule).properties(title='Mastery Probability Distribution')

alt.LayerChart(...)

## KC dfficulty ranking


If we want to rank the KC difficulty by showing the % of the class that is below a certain threshold

In [204]:
kc_difficulty = (
    initial_model_output
    .groupby('skill_name')['state_predictions']
    .apply(lambda x: (x < 0.95).mean() * 100)
    .reset_index()
    .rename(columns={'state_predictions': 'pct_not_mastered'})
    .sort_values('pct_not_mastered', ascending=False)
)
kc_difficulty

,skill_name,pct_not_mastered
0,MKC.U1.01.data_context_variables,100.000000
35,MKC.U7.01.t_intervals_mean_foundations,100.000000
26,MKC.U5.02.sample_proportion_sampling_distribution,100.000000
27,MKC.U5.03.diff_proportions_sampling_distribution,100.000000
28,MKC.U5.04.sample_mean_sampling_distribution,100.000000
29,MKC.U5.05.diff_means_sampling_distribution,100.000000
31,MKC.U6.02.one_prop_hypothesis_tests,100.000000
32,MKC.U6.03.errors_alpha_power,100.000000
33,MKC.U6.04.two_prop_confidence_intervals,100.000000
34,MKC.U6.05.two_prop_hypothesis_tests,100.000000


Top 10 KCs with lowest mastery :

In [205]:
threshold = 0.7
kc_difficulty = (
    initial_model_output
    .groupby('skill_name')['state_predictions']
    .mean()
    .reset_index()
    .rename(columns={'state_predictions': 'avg_mastery'})
    .sort_values('avg_mastery')
)
kc_difficulty

,skill_name,avg_mastery
7,MKC.U10.02.bootstrap_intervals,0.029012
18,MKC.U3.04.random_assignment_causation_design,0.059881
40,MKC.U7.06.multiple_comparisons_family_error,0.065202
2,MKC.U1.03.quantitative_displays_description,0.096238
35,MKC.U7.01.t_intervals_mean_foundations,0.135633
10,MKC.U2.01.two_way_categorical_association,0.144931
21,MKC.U4.02.conditional_compound_independence,0.149066
17,MKC.U3.03.study_type_variables_units,0.159712
19,MKC.U3.05.blocking_matching_blinding_confounding,0.199109
37,MKC.U7.03.one_sample_t_tests,0.214692


In [206]:
alt.Chart(kc_difficulty[:10]).mark_bar().encode(
    x=alt.X('avg_mastery', title = 'Average mastery'),
    y=alt.Y('skill_name', sort='x', title = 'Skill name'),
    tooltip=alt.Tooltip(['avg_mastery'], title = 'Average mastery')
).properties(
    title = 'Top 10 KCs with lowest Mastery Probability',
)

alt.Chart(...)

In [207]:
alt.Chart(kc_difficulty[-10:]).mark_bar().encode(
    x=alt.X('avg_mastery', title = 'Average mastery'),
    y=alt.Y('skill_name', sort='x', title = 'Skill name'),
    tooltip=alt.Tooltip(['avg_mastery'], title = 'Average mastery')
).properties(
    title = 'Top 10 KCs with highest Mastery Probability',
)

alt.Chart(...)

In [208]:
initial_model_output[initial_model_output['skill_name']=='KC.U5.22.difference_sample_means_standard_error']

,Unnamed: 0,user_id,skill_name,correct,order_id,correct_predictions,state_predictions


Notes : maybe only keep the latest state_prediction in the case that there are multiple

## KC progress

In [209]:
initial_model_output = initial_model_output.sort_values(['user_id', 'skill_name', 'order_id'])
initial_model_output['kc_attempt'] = (
    initial_model_output
    .groupby(['user_id', 'skill_name'])
    .cumcount() + 1
)

In [225]:
initial_model_output[initial_model_output['skill_name']=='MKC.U6.02.one_prop_hypothesis_tests'].sort_values('kc_attempt')

,Unnamed: 0,user_id,skill_name,correct,order_id,correct_predictions,state_predictions,kc_attempt
3,96,S001,MKC.U6.02.one_prop_hypothesis_tests,0,3,0.393887,0.300741,1
666,1478,S009,MKC.U6.02.one_prop_hypothesis_tests,0,56,0.393887,0.300741,1
697,74,S010,MKC.U6.02.one_prop_hypothesis_tests,1,4,0.393887,0.300741,1
784,176,S011,MKC.U6.02.one_prop_hypothesis_tests,1,5,0.393887,0.300741,1
896,850,S012,MKC.U6.02.one_prop_hypothesis_tests,0,25,0.393887,0.300741,1
...,...,...,...,...,...,...,...,...
373,1898,S005,MKC.U6.02.one_prop_hypothesis_tests,0,70,0.507600,0.721965,7
1025,1895,S013,MKC.U6.02.one_prop_hypothesis_tests,1,87,0.539446,0.839927,8
1547,1708,S020,MKC.U6.02.one_prop_hypothesis_tests,1,82,0.364475,0.191794,8
1026,1896,S013,MKC.U6.02.one_prop_hypothesis_tests,0,88,0.558058,0.908873,9


In [214]:
# Then average across students for each KC at each attempt number
mastery_over_time = (
    initial_model_output
    .groupby(['kc_attempt', 'skill_name'])['state_predictions']
    .mean()
    .reset_index()
    .rename(columns={'state_predictions': 'avg_mastery'})
)

In [ ]:
initial_model_output = initial_model_output.sort_values(['user_id', 'skill_name', 'order_id'])
initial_model_output['kc_attempt'] = (
    initial_model_output
    .groupby(['user_id', 'skill_name'])
    .cumcount() + 1
)
# Then average across students for each KC at each attempt number
mastery_over_time = (
    initial_model_output
    .groupby(['kc_attempt', 'skill_name'])['state_predictions']
    .mean()
    .reset_index()
    .rename(columns={'state_predictions': 'avg_mastery'})
)
kc_selection = alt.selection_point(fields=['skill_name'], bind='legend')

alt.Chart(mastery_over_time).mark_line(point=True).encode(
    x=alt.X('kc_attempt:Q', title='Attempt number (per KC)'),
    y=alt.Y('avg_mastery:Q', title='Average mastery', scale=alt.Scale(domain=[0, 1])),
    color=alt.Color('skill_name:N', legend=alt.Legend(title='KC')),
    opacity=alt.condition(kc_selection, alt.value(1), alt.value(0.1)),
    tooltip=[
        alt.Tooltip('skill_name:N', title='KC'),
        alt.Tooltip('kc_attempt:Q', title='Attempt'),
        alt.Tooltip('avg_mastery:Q', title='Avg mastery', format='.3f')
    ]
).add_params(kc_selection).properties(
    title='Mastery over time per KC',
    width=500, height=300
)

alt.Chart(...)

In [228]:
kc_selection = alt.selection_point(fields=['skill_name'], bind='legend')

alt.Chart(mastery_over_time[mastery_over_time['skill_name']=='MKC.U6.02.one_prop_hypothesis_tests']).mark_line(point=True).encode(
    x=alt.X('kc_attempt:Q', title='Attempt number (per KC)'),
    y=alt.Y('avg_mastery:Q', title='Average mastery', scale=alt.Scale(domain=[0, 1])),
    tooltip=[
        alt.Tooltip('skill_name:N', title='KC'),
        alt.Tooltip('kc_attempt:Q', title='Attempt'),
        alt.Tooltip('avg_mastery:Q', title='Avg mastery', format='.3f')
    ]
).properties(
    title='Mastery over time per KC',
    width=500, height=300
)

alt.Chart(...)

In [246]:
first_attempt = initial_model_output.groupby(['user_id','skill_name'])['state_predictions'].first().mean()
last_attempt = initial_model_output.groupby(['user_id','skill_name'])['state_predictions'].last().mean()

print(f"Avg mastery at first attempt: {first_attempt:.3f}")
print(f"Avg mastery at latest attempt: {last_attempt:.3f}")


Avg mastery at first attempt: 0.378
Avg mastery at latest attempt: 0.431
